# Model Notebook
### Emotion Classification – Base Models + Stacking Ensemble

**Prerequisites:** Run `notebooks/preprocess.ipynb` first to generate `datasets/processed/`.

**Flow:**
1. Load processed data & config
2. Base model A – **Naive Bayes** (TF-IDF + MultinomialNB)
3. Base model B – **SVM** (TF-IDF + LinearSVC w/ calibration)
4. Base model C – **XGBoost** (TF-IDF + XGBClassifier)
5. **Stacking Ensemble** (out-of-fold → Logistic Regression meta-learner)
6. All models saved to `models/`

## 1. Load Processed Data & Configuration

In [ ]:
import os, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.preprocessing import LabelEncoder

# ---- Load config written by preprocess.ipynb ----
PROCESSED_DIR = os.path.join("datasets", "processed")
with open(os.path.join(PROCESSED_DIR, "config.json")) as f:
    cfg = json.load(f)

EMOTION_LABELS = cfg["EMOTION_LABELS"]
N_FOLDS        = cfg["N_FOLDS"]
SEED           = cfg["SEED"]
K              = cfg["K"]
MODELS_DIR     = cfg["MODELS_DIR"]

os.makedirs(MODELS_DIR, exist_ok=True)

# ---- Load arrays ----
X_train = np.load(os.path.join(PROCESSED_DIR, "X_train.npy"), allow_pickle=True)
X_test  = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"),  allow_pickle=True)
y_train = np.load(os.path.join(PROCESSED_DIR, "y_train.npy"))
y_test  = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))
classes = np.load(os.path.join(PROCESSED_DIR, "label_encoder_classes.npy"), allow_pickle=True)

le = LabelEncoder()
le.classes_ = classes

print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Classes ({K}):", list(le.classes_))
print(f"Emotion labels: {EMOTION_LABELS}")
print(f"N_FOLDS={N_FOLDS}  SEED={SEED}  MODELS_DIR='{MODELS_DIR}'")

## Helper – Confusion Matrix Plotter

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix", figsize=(8, 6)):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(K)))
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set(xticks=range(K), yticks=range(K),
           xticklabels=le.classes_, yticklabels=le.classes_,
           xlabel="Predicted label", ylabel="True label",
           title=title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    thresh = cm.max() / 2.0
    for i in range(K):
        for j in range(K):
            ax.text(j, i, f"{cm[i, j]}",
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.show()


def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    print(f"=== {name} ===")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1-macro : {f1:.4f}")
    print()
    print(classification_report(y_true, y_pred,
                                 target_names=le.classes_,
                                 zero_division=0))
    plot_confusion_matrix(y_true, y_pred, title=f"Confusion Matrix – {name}")

print("Helper functions ready.")

## 2. Base Model A – Naive Bayes

In [ ]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

def make_nb():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf",   MultinomialNB(alpha=0.5)),
    ])

nb_model = make_nb()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

# Save
nb_path = os.path.join(MODELS_DIR, "naive_bayes.pkl")
joblib.dump(nb_model, nb_path)
print(f"Naive Bayes model saved to '{nb_path}'")

evaluate("Naive Bayes", y_test, y_pred_nb)

## 3. Base Model B – SVM

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

def make_svm():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf",   CalibratedClassifierCV(LinearSVC(class_weight="balanced"), cv=3)),
    ])

svm_model = make_svm()
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

# Save
svm_path = os.path.join(MODELS_DIR, "svm.pkl")
joblib.dump(svm_model, svm_path)
print(f"SVM model saved to '{svm_path}'")

evaluate("SVM", y_test, y_pred_svm)

## 4. Base Model C – XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from scipy.sparse import issparse

# Wrap XGBoost in a pipeline-compatible estimator
# XGBClassifier needs numeric labels starting at 0
def make_xgb():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf",   XGBClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            use_label_encoder=False,
            eval_metric="mlogloss",
            random_state=SEED,
            n_jobs=-1,
        )),
    ])

xgb_model = make_xgb()
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

# Save
xgb_path = os.path.join(MODELS_DIR, "xgboost.pkl")
joblib.dump(xgb_model, xgb_path)
print(f"XGBoost model saved to '{xgb_path}'")

evaluate("XGBoost", y_test, y_pred_xgb)

## 5. Stacking Ensemble

**Strategy:** Out-of-fold (OOF) predictions from the three base models become
features for a Logistic Regression meta-learner.  This avoids data leakage
because each training fold's predictions are made by a model that never saw
those samples during training.

```
              [NB proba] [SVM proba] [XGB proba]
                         ↓
              Logistic Regression (meta-learner)
                         ↓
                  Emotion prediction
```

### 5.1 Generate Out-of-Fold Predictions

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
n_train = len(X_train)

oof = {
    "nb":  np.zeros((n_train, K)),
    "svm": np.zeros((n_train, K)),
    "xgb": np.zeros((n_train, K)),
}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), 1):
    print(f"Fold {fold}/{N_FOLDS}")
    Xtr, Xva = X_train[tr_idx], X_train[va_idx]
    ytr = y_train[tr_idx]

    nb_f  = make_nb().fit(Xtr, ytr)
    svm_f = make_svm().fit(Xtr, ytr)
    xgb_f = make_xgb().fit(Xtr, ytr)

    oof["nb"][va_idx]  = nb_f.predict_proba(Xva)
    oof["svm"][va_idx] = svm_f.predict_proba(Xva)
    oof["xgb"][va_idx] = xgb_f.predict_proba(Xva)

print("OOF prediction complete.")

### 5.2 Train Meta-Learner

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_train = np.hstack([oof["nb"], oof["svm"], oof["xgb"]])

meta_learner = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED,
)
meta_learner.fit(meta_train, y_train)
print("Meta-learner (Logistic Regression) trained.")

### 5.3 Evaluate Stacking Ensemble on Test Set

In [ ]:
# Generate test-set probabilities from each base model (trained on full train set)
p_nb  = nb_model.predict_proba(X_test)
p_svm = svm_model.predict_proba(X_test)
p_xgb = xgb_model.predict_proba(X_test)

meta_test  = np.hstack([p_nb, p_svm, p_xgb])
y_pred_ens = meta_learner.predict(meta_test)

evaluate("Stacking Ensemble (NB + SVM + XGBoost)", y_test, y_pred_ens)

## 6. Save All Models

In [ ]:
# Base models were already saved individually above.
# Save the meta-learner.
meta_path = os.path.join(MODELS_DIR, "meta_learner.pkl")
joblib.dump(meta_learner, meta_path)

# Save the label-encoder classes so inference code can reconstruct it
le_path = os.path.join(MODELS_DIR, "label_encoder_classes.npy")
np.save(le_path, le.classes_)

print("All models saved:")
for fname in ["naive_bayes.pkl", "svm.pkl", "xgboost.pkl",
              "meta_learner.pkl", "label_encoder_classes.npy"]:
    fpath = os.path.join(MODELS_DIR, fname)
    size  = os.path.getsize(fpath) / 1024
    print(f"  {fpath}  ({size:.1f} KB)")

## 7. Summary Comparison

In [ ]:
import pandas as pd

results = []
for name, y_pred in [
    ("Naive Bayes", y_pred_nb),
    ("SVM",         y_pred_svm),
    ("XGBoost",     y_pred_xgb),
    ("Ensemble",    y_pred_ens),
]:
    results.append({
        "Model":    name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "F1-macro": round(f1_score(y_test, y_pred, average="macro", zero_division=0), 4),
    })

summary = pd.DataFrame(results).set_index("Model")
print(summary.to_string())
summary